In [2]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/enrichment_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/gene_mapping_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/top_corr_module_fxns.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: future


Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths



Attaching package: ‘data.table’


The following objects are masked from ‘package:reshape2’:

    dcast, melt


The following objects are masked from ‘package:dplyr’:

    between, first, last




Here I perform enrichment analysis to find modules enriched for cell type markers. 

These modules will later be used to correlate with exon PSI to define cell type-specific exons.

In [ ]:
n_workers <- 20
mod_def <- "PosBC"
max_set_size <- 200
min_set_size <- 3

## Prep marker genes

### Claude marker genes

In [3]:
set_source <- "Claude_marker_genes"

marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")

### Broad 

In [ ]:
library(msigdbr)

In [ ]:
gs_GOBP <- msigdbr(collection="C5", subcollection="GO:BP")

gs_CTS <- msigdbr(collection="C8")
gs_CTS <- gs_CTS[gs_CTS$gs_geoid == "GSE104276",]

gs_GOBP_list <- tapply(gs_GOBP$gene_symbol, INDEX=gs_GOBP$gs_name, FUN=list)
gs_CTS_list <- tapply(gs_CTS$gene_symbol, INDEX=gs_CTS$gs_name, FUN=list)

gs_list <- c(gs_GOBP_list, gs_CTS_list)

In [ ]:
set_sizes <- lengths(gs_list)
mask <- (set_sizes < max_set_size) & (set_sizes > min_set_size)
gs_list <- gs_list[mask]

print(length(gs_list))

In [ ]:
marker_genes_list <- gs_list

In [ ]:
set_source <- paste0("MSigDB_", length(gs_list), "sets")

### MO marker genes

In [ ]:
load("/mnt/lareaulab/reliscu/data/gene_sets/Oldham/MO_sets.RData")

mask <- !MO_legend$Category %in% "Disease"
marker_genes_list <- MO_sets[mask]
names(marker_genes_list) <- MO_legend$SetName[mask]

set_sizes <- lengths(marker_genes_list)
mask <- (set_sizes < max_set_size) & (set_sizes > min_set_size)
marker_genes_list <- marker_genes_list[mask]

print(length(marker_genes_list))

set_source <- paste0("MO_", length(marker_genes_list), "sets")

## Get enrichments

In [ ]:
network_dir <- "GTEx_cortex_counts_All_501_outliers_removed_TMM_ComBat_SMGEBTCH_corrected_All_370_outliers_removed_mergeParam0.95_subsetCutoff11.245_Modules"

In [ ]:
enrichments_df <- get_module_enrichments_par(network_dir, marker_genes_list, mod_def, n_workers)

write.csv(enrichments_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments.csv"), row.names=FALSE, quote=FALSE)

# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)
    
# top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

# get_mod_vs_DE_genes(top_mods_df, marker_genes_list, mod_def)

write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments_top_Qval_mods.csv"), row.names=FALSE, quote=FALSE)

## Compare enrichments

In [5]:
x <- list.files("data/enrichments")[grep("Qval", list.files("data/enrichments"))]
x <- x[grep("MO", x)]

enrichments_list <- lapply(x, function(y) {
    split <- unlist(strsplit(y, "STAR_"))
    prefix <- unlist(strsplit(split[1], "_ComBat"))[1]
    middle <- gsub("p_", "p", unlist(strsplit(split[1], "yao_2021_"))[2])
    suffix <- gsub("_enrichments.csv", "", unlist(strsplit(split[2], "DE_genes_"))[2])
    dataname <- paste(prefix, paste(middle, suffix, sep="_"), sep="_")

    enrich_df <- fread(file.path("data/enrichments", y), data.table=FALSE)
    enrich_df <- enrich_df[,c("Cell_type", "Qval")]
    colnames(enrich_df)[2] <- dataname
    enrich_df
})

enrichment_table <- Reduce(f=function(x, y) {
    merge(x, y, by="Cell_type", all=TRUE)
}, enrichments_list)

Warning message in fread(file.path("data/enrichments", y), data.table = FALSE):
“Previous fread() session was not cleaned up properly. Cleaned up ok at the beginning of this fread() call.”


In [6]:
enrichment_table

Cell_type,GTEx_cortex_counts_All_501_outliers_removed_TMM_NA_NA,GTEx_cortex_counts_TMMF_All_501_outliers_removed_mergeParam0.95_subsetCutoff10.171_Modules_MO_1271sets_enrichments_top_Qval_mods.csv_NA_NA
<chr>,<dbl>,<dbl>
ABA_ASTROCYTES,1.133512e-11,3.678944e-14
ABA_NEURONS,NA,6.930343e-18
ABSINTA_2021_MS_OPC_DE_OG_CLUSTERS,2.207872e-03,NA
ABSINTA_2021_PLASMABLASTS_DE_MG_CLUSTERS,2.441422e-13,NA
ABSINTA_2021_VASCULAR_UNKNOWN_DE_VASC_CLUSTERS,NA,1.132847e-42
AGARWAL_CTX_2020_IN1A_DE_IN_CLUSTERS,1.975611e-21,3.607504e-29
AGARWAL_CTX_2020_IN1B,5.745537e-53,NA
AGARWAL_CTX_2020_IN4A_DE_IN_CLUSTERS,NA,7.707139e-16
AGARWAL_SN_2020_ASTROCYTES_2_DE_ASC_CLUSTERS,NA,6.481391e-09


In [7]:
data.frame(colMeans(enrichment_table[,-1], na.rm=TRUE))

,colMeans.enrichment_table....1...na.rm...TRUE.
,<dbl>
GTEx_cortex_counts_All_501_outliers_removed_TMM_NA_NA,0.05566010
GTEx_cortex_counts_TMMF_All_501_outliers_removed_mergeParam0.95_subsetCutoff10.171_Modules_MO_1271sets_enrichments_top_Qval_mods.csv_NA_NA,0.04592989
